# Stage 6: Retrieval Add-on (Optional Ensemble Component)
This notebook tests an external retrieval dataset HealthCareMagic-100k mapped to back-translated Bengali queries to explore blending retrieval signals.

### 1. Install & Load Dependencies

In [ ]:
%pip install -q datasets sentence-transformers pandas transformers torch

import os
import sys
import pandas as pd
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util

print("Dependencies loaded successfully.")

### 2. Load HealthCareMagic English Dataset

In [ ]:
print("Loading HealthCareMagic dataset...")
ds_en = load_dataset("lavita/medical-qa-datasets", "chatdoctor_healthcaremagic")
split_name = list(ds_en.keys())[0]
df_en = ds_en[split_name].to_pandas()
print("English Dataset Schema Discovery:")
print(df_en.head(3))

# Verify the columns
q_col, a_col = "input", "output"
if "input" not in df_en.columns or "output" not in df_en.columns:
    # Auto detect columns
    q_col = [c for c in df_en.columns if "question" in c.lower() or "input" in c.lower()][0]
    a_col = [c for c in df_en.columns if "answer" in c.lower() or "output" in c.lower()][0]
print(f"Detected columns: question='{q_col}', answer='{a_col}'")

### 3. Setup Translation Pipeline (Bengali to English)

In [ ]:
from transformers import pipeline

# Load translation model
translator = pipeline(
    "translation", 
    model="facebook/nllb-200-distilled-600M", 
    src_lang="ben_Beng", 
    tgt_lang="eng_Latn",
    device=0 if torch.cuda.is_available() else -1
)
print("Translator loaded.")

### 4. Back-translate Validation Queries and Embed

In [ ]:
val_df = pd.read_csv("/kaggle/working/sft_val.csv")
# Limit to 50 rows for validation checks
val_subset = val_df.head(50).copy()

print("Translating queries Bengali -> English...")
translated_inputs = []
for text in val_subset["input"]:
    res = translator(text, max_length=512)
    translated_inputs.append(res[0]['translation_text'])
val_subset["translated_input"] = translated_inputs
print("Example translation:", val_subset.iloc[0]["translated_input"])

### 5. Semantic Search & Cosine Similarity Sweep

In [ ]:
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

print("Embedding English corpus questions...")
# Sample first 10,000 for fast demonstration run
corpus_questions = df_en[q_col].head(10000).tolist()
corpus_answers = df_en[a_col].head(10000).tolist()
corpus_embeddings = embedder.encode(corpus_questions, convert_to_tensor=True)

print("Embedding back-translated queries...")
query_embeddings = embedder.encode(val_subset["translated_input"].tolist(), convert_to_tensor=True)

# Compute cosine similarities
cosine_scores = util.cos_sim(query_embeddings, corpus_embeddings)

# Find best matching translation for answers
translator_en_to_bn = pipeline(
    "translation", 
    model="facebook/nllb-200-distilled-600M", 
    src_lang="eng_Latn", 
    tgt_lang="ben_Beng",
    device=0 if torch.cuda.is_available() else -1
)

best_match_indices = []
best_match_scores = []
for i in range(len(val_subset)):
    best_idx = int(np.argmax(cosine_scores[i].cpu().numpy()))
    best_match_indices.append(best_idx)
    best_match_scores.append(float(cosine_scores[i][best_idx].cpu().numpy()))

val_subset["retrieved_idx"] = best_match_indices
val_subset["similarity"] = best_match_scores
print("Max similarity scores:", val_subset["similarity"].describe())

### 6. Sweep Threshold Evaluation

In [ ]:
sys.path.append(os.path.abspath('src'))
import metric_utils

# Translate retrieved answers to Bengali
retrieved_answers_bn = []
for idx in best_match_indices:
    eng_ans = corpus_answers[idx]
    res = translator_en_to_bn(eng_ans[:400], max_length=512)
    retrieved_answers_bn.append(res[0]['translation_text'])
val_subset["retrieved_answer_bn"] = retrieved_answers_bn

# Score retrieval answers directly
scores = []
for _, row in val_subset.iterrows():
    score, _ = metric_utils.composite_score([row["retrieved_answer_bn"]], [row["output"]])
    scores.append(score)
val_subset["retrieval_score"] = scores

# Check average score vs threshold
thresholds = [0.75, 0.80, 0.85, 0.90, 0.95]
print("\n=== Cosine Similarity Threshold Sweep ===")
for t in thresholds:
    subset_above = val_subset[val_subset["similarity"] >= t]
    num_rows = len(subset_above)
    if num_rows > 0:
        mean_ret_score = subset_above["retrieval_score"].mean()
        print(f"Threshold >= {t:.2f} | Rows above: {num_rows} | Mean Retrieval Score: {mean_ret_score:.4f}")
    else:
        print(f"Threshold >= {t:.2f} | Rows above: 0 | Mean Retrieval Score: N/A")

### 7. Verdict Output

In [ ]:
retrieval_improves = False
val_subset_above = val_subset[val_subset["similarity"] >= 0.85]
if len(val_subset_above) > 0:
    # If retrieval matches score above model's average (~0.45 typical validation score)
    if val_subset_above["retrieval_score"].mean() > 0.45:
        retrieval_improves = True

if retrieval_improves:
    print("Verdict: retrieval add-on IMPROVED local validation score. Recommend USING it for final blended submission.")
else:
    print("Verdict: retrieval add-on DID NOT IMPROVE local validation score. Recommend NOT USING it for the final submission.")